In [1]:
TECHSTORE_POLICY = """
TECHSTORE CUSTOMER SUPPORT POLICY

Company:
TechStore is an online electronics store selling laptops, phones,
tablets, monitors, keyboards, mice, headphones, and other accessories.

SHIPPING
- Standard shipping takes 3-5 business days.
- Express shipping takes 1-2 business days.
- Orders are processed within 1 business day.
- Customers receive a tracking number after the order ships.
- We currently ship within India.

RETURNS
- Customers can request a return within 30 days of delivery.
- Products must be unused and in their original packaging.
- Damaged or defective products can also be returned.
- Refunds are normally processed within 5-7 business days after
  the returned product is inspected.

WARRANTY
- Electronics purchased from TechStore include a 1-year limited warranty.
- The warranty covers manufacturing defects.
- Accidental damage is not covered.
- Physical damage caused by misuse is not covered.

CANCELLATIONS
- Orders can be cancelled before they are shipped.
- Once an order has shipped, it cannot be cancelled.
- Customers can request a return after delivery if the order qualifies.

PAYMENTS
- TechStore accepts credit cards, debit cards, UPI, and net banking.
- Payment information should never be requested from customers.

ORDER STATUS
- The AI assistant does NOT have access to real customer orders,
  order numbers, payment information, or shipping systems.
- Never invent an order status.
- If a customer asks about a specific order, explain that the assistant
  cannot access live order information and direct the customer to
  contact TechStore support.

CUSTOMER SUPPORT
- Be friendly, professional, concise, and helpful.
- Answer using the TechStore policies above.
- Never invent a policy that is not provided above.
- If the policy does not contain the answer, say that you do not have
  enough information rather than guessing.
"""

In [2]:
BUSINESS_SYSTEM_PROMPT = f"""
You are the customer support AI assistant for TechStore,
an online electronics store.

Your job is to help customers with questions about:
- shipping
- returns
- refunds
- warranties
- cancellations
- payments
- general TechStore policies

Follow the company policy exactly.

Important rules:

1. Never invent information.
2. Never claim that you accessed an order, database, or tracking system.
3. Never ask customers for passwords, credit card numbers, CVV codes,
   OTPs, or other sensitive payment information.
4. If a customer asks about a specific order, explain that you cannot
   access live order information.
5. If something is not covered by the policy, clearly say so.
6. Be friendly and professional.
7. Keep normal answers concise.
8. Explain policies clearly when customers need more detail.

Here is the official TechStore policy:

{TECHSTORE_POLICY}
"""

In [3]:
import os 
import time

from dotenv import load_dotenv
from google import genai
import gradio as gr
from google.genai import types
import ollama

In [4]:
load_dotenv(override=True)
gemini_api_key = os.getenv('GOOGLE_API_KEY')

if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")



Gemini API Key exists and begins AQ.Ab8RN


In [5]:
MODEL = 'gemini-2.5-flash'

gemini_client = genai.Client(api_key=gemini_api_key)

ollama_client = ollama.Client(host='http://localhost:11434')

In [6]:
from pathlib import Path

KB_PATH = Path('knowledge_base')

documents = []

for file_path in KB_PATH.glob('*.txt'):
    text = file_path.read_text(encoding='utf-8')

    documents.append({
        'source': file_path.name,
        'text': text,
    })

print(f"Loaded {len(documents)} documents")

for document in documents:
    print(document['source'])



Loaded 0 documents


In [7]:
embedding = ollama_client.embed(
    model='nomic-embed-text',
    input='How long does shipping take?'
)

print(len(embedding['embeddings'][0]))

768


In [8]:
templates = {
    'Helpful Assistant': 'You are a helpful assistant.',
    'Python Tutor': 'Explain Python clearly with examples.',
    'Translator-French': 'Translate everything into French.',
    'Travel Guide': 'Act as an experienced travel guide.'
}   

In [9]:
def chat(
    message,
    history,
    model,
    temperature,
    system_prompt,
):
    start = time.time()

    provider, model_name = model.split(": ", 1)

    try:

        # --------------------------------------------------
        # GEMINI
        # --------------------------------------------------
        if provider == "Gemini":

            contents = []

            for msg in history:

                role = "model" if msg['role'] == 'assistant' else "user"

                contents.append({
                    "role": role,
                    "parts": [
                        {
                            "text": msg["content"]
                        }
                    ]
                })

            # Add current user message
            contents.append({
                "role": "user",
                "parts": [
                    {
                        "text": message
                    }
                ]
            })

            response_text = ""

            stream = gemini_client.models.generate_content_stream(
                model=model_name,
                contents=contents,
                config=types.GenerateContentConfig(
                    temperature=temperature,
                    system_instruction=system_prompt,
                ),
            )

            for chunk in stream:

                if chunk.text:
                    response_text += chunk.text

                    yield response_text

        # --------------------------------------------------
        # OLLAMA
        # --------------------------------------------------
        elif provider == "Ollama":

            messages = [
                {
                    "role": "system",
                    "content": system_prompt
                }
            ]

            for msg in history:

                messages.append({
                    "role": msg["role"],
                    "content": msg["content"]
                })

            # Add current user message
            messages.append({
                "role": "user",
                "content": message
            })

            response_text = ""

            stream = ollama_client.chat(
                model=model_name,
                messages=messages,
                options={
                    "temperature": temperature
                },
                stream=True
            )

            for chunk in stream:

                content = chunk["message"]["content"]

                if content:
                    response_text += content

                    yield response_text

        else:

            yield "⚠️ Unknown model provider."

        end = time.time()

        print(
            f"{provider} | "
            f"{model_name} | "
            f"Response time: {end - start:.2f} seconds"
        )

    except Exception as e:

        yield f"⚠️ Error: {e}"

In [10]:
system_prompts = {
    "General Assistant": "You are a helpful and friendly AI assistant. Give clear, accurate, and concise answers.",
    "Coding Assistant": "You are an expert Python programmer. Explain solutions clearly, write clean and efficient code, and point out common mistakes.",
    "Teacher": "You are a patient teacher. Explain concepts in simple terms, use examples, and gradually increase the level of difficulty.",
    "LeetCode Coach": "You are a coding interview coach. Help the user solve algorithm problems. Do not immediately give the complete solution. First provide hints and guide the user toward the solution.",
    "Code Reviewer": "You are a senior software engineer reviewing code. Identify bugs, performance issues, readability problems, and suggest improvements.",
    "Concise Assistant": "Answer in no more than 3 sentences unless the user asks for more detail.",
    "Detailed Explainer": "You are an expert educator. Explain concepts step by step using simple language, intuitive examples, and analogies where appropriate.",
    "Socratic Tutor": "Teach using the Socratic method. Instead of immediately giving the answer, ask questions that help the user discover the answer themselves.",
}




In [11]:
css = """
#system-prompt textarea {
    max-height: 300px !important;
    min-height: 300px !important;
    overflow-y: auto !important;
    overflow-x: hidden !important;
    resize: vertical !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Glass(), 
    title='TechStore Customer Support AI',
    css=css
) as demo:
    gr.Markdown("""
            # 🛒 TechStore Customer Support AI

AI-powered customer support for TechStore.

Ask questions about shipping, returns, refunds, warranties,
cancellations, and other company policies.
        """)

    with gr.Row():
        model = gr.Dropdown(
            choices=[
                "Gemini: gemini-2.5-flash",
                "Gemini: gemini-2.5-pro",
                "Ollama: llama3.2:3b",
                "Ollama: qwen3:8b",
            ],
            value="Gemini: gemini-2.5-flash",
            label="Model",
            interactive=True
        )

        temperature = gr.Slider(
            minimum=0,
            maximum=2,
            step=0.1,
            value=0.7,
            label='Temperature',
            interactive=True
        )

    with gr.Accordion("Advanced Settings", open=False):

        system_prompt = gr.Textbox(
            label="System Prompt",
            lines=10,
            max_lines=25,
            value=BUSINESS_SYSTEM_PROMPT,
            interactive=True,
            elem_id = 'system-prompt'
        ) 

    with gr.Accordion("TechStore company policy", open = False):
        gr.Markdown(TECHSTORE_POLICY)

    gr.ChatInterface(
        fn=chat,
        type="messages",
        additional_inputs=[
            model,
            temperature,
            system_prompt,
        ],
        examples=[
            ["Can I return a laptop after 20 days?", None, None, None],
            ["How long does standard shipping take?", None, None, None],
            ["Does TechStore provide a warranty?", None, None, None],
            ["Can I cancel my order after it has shipped?", None, None, None],
            ["What payment methods do you accept?", None, None, None],
            ["Where is my order?", None, None, None],
            ["My laptop arrived damaged. What should I do?", None, None, None],
        ],
    )

In [ ]:

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Gemini | gemini-2.5-flash | Response time: 6.92 seconds
Ollama | qwen3:8b | Response time: 88.66 seconds
